In [ ]:
# NOTE : 02_A_SISaP.ipynb - Cell 1 (one cell: photometry for every star, then the SISaP-over-DASCH plots)
# Generalized multi-star SISaP, V CrA recipe. Runs every star folder under BASE_DATA_DIR on the DASCH-accepted plates only.
# Plates: the manifest written by dasch_Cutout_Recreation (dasch_accepted == True and status == downloaded), or, for an older folder
#         without that flag, the accepted plates in dasch_official_lightcurve.csv.
# Per plate the target is measured two ways from the same detections, then the two are merged using only our own numbers:
#   isophotal flux (best for faint targets) and a 12 px circle (best for bright and middle targets, much more repeatable)
#   circle value when it is a measurement, its bright-end slope is sane and it is brighter than ISO_FROM_MAG, otherwise the isophotal value
# Bright targets (SAT_EXTEND): when the target is brighter than the unsaturated calibrators, the saturated calibrators are added to the curve
#   and the isophotal value is used (the circle is cut off by a big image). Such points are tier 'uncertain', flag 'sat_ext', with an error from
#   leaving calibrators out of the fit. A target brighter than every calibrator stays a limit. Tested on simulated saturated plates only.
# Edge of the range (EDGE_EXTEND): a target up to 0.6 instrumental mag brighter than the brightest calibrator is kept (isophotal, 'uncertain', flag 'edge',
#   error + 0.7 mag or more). Further out it stays a limit; extrapolating there was off by 1.5-3 mag against DASCH. A cutout cannot calibrate a bright star in a sparse field.
# Reference catalogue (REF_CATALOG): 'gaia' = Gaia DR3 (G, BP, RP from GAVO) turned into V and B, like DASCH's Gaia-based reference, with a blend test so that
#   calibrators with a close neighbour are not used; 'apass' = APASS DR10. Gaia results go to <star>/02_A_gaia, APASS results stay in <star>/02_A.
# Needs: sep, astropy, scipy, pandas, matplotlib (statsmodels is optional). No daschlab and no photutils in this cell.
# Per star it writes, in <star>/02_A_gaia (REF_CATALOG = 'gaia') or <star>/02_A ('apass'); the plots come after the photometry, in the same cell:
#   lightcurve.csv (jd, magnitude, magnitude_error, then extra columns), limits.csv, photometry_results.csv, gaia_field_raw.csv or apass_field.csv, run_summary.txt
# Re-running resumes: finished plates are kept in <that folder>/algorithm_cache_sep

import os, io, re, math, time, shelve, hashlib, traceback, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import urllib.request, urllib.parse
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.time import Time
import astropy.units as u
from scipy import stats
from scipy.spatial import cKDTree
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings('ignore')

try:
    import sep
except ImportError:
    raise SystemExit('The sep package is not installed in this kernel. Run   pip install sep   in a cell, restart the kernel, and run this cell again.')


# SETTINGS
BASE_DATA_DIR = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data")
TARGETS_FILE_NAME = 'rcb_Bright_Targets.txt'   # optional: "Name|RA_deg|Dec_deg" lines, same file the cutout notebook reads
STARS = None                  # None = every star folder with a cutouts/ directory; or a list of folder names, e.g. ['V_CrA']
N_PLATES = None               # None = all plates of each star; a number = only the first N (a quick test)
ONLY_DASCH_ACCEPTED = True    # True: only the plates DASCH accepted; False: every cutout in the folder
PLATE_LIMIT_PER_STAR = N_PLATES
MAKE_PLOTS = True             # True: after the photometry, draw SISaP (blue, top) over DASCH (black, bottom) for every star
DASCH_ERR_COL = 'magcal_local_error'   # DASCH error column; other options: 'magcal_local_rms', 'magcal_magdep_rms'
SHOW_LIMITS = False           # True: SISaP limits as open blue triangles on the top panel

# Target coordinates: a star listed here uses this position; any other star uses its line in the targets file, then <star>/target_coord.txt
# (written by the cutout notebook), then a SIMBAD/Sesame
# lookup of the folder name (underscores read as spaces; saved in <star>/02_A/target_coord.txt), then the centre of its cutouts.
MANUAL_TARGET_COORDS = {
    'V_CrA': SkyCoord(ra=281.884623417 * u.deg, dec=-38.158974417 * u.deg),
}
# Plates thrown out by eye, per star folder (file names). A text file <star>/02_A/manual_reject.txt (one name per line) is read too.
MANUAL_REJECT = {'V_CrA': ['00859_b37936m0s0.fits']}

# Merge of the two flux measurements
APER_R_PX = 12.0              # radius of the circle (px; 1 px = 1.44 arcsec)
ISO_FROM_MAG = 13.0           # circle magnitude at or fainter than this: use the isophotal value
BAD_SLOPE = 0.7               # circle result with a bright-end calibration slope below this is not used
FALLBACK_LIMITS = True        # circle gives only a limit, isophotal has a measurement: use the isophotal one

# Calibration and target pick (as validated on V CrA)
EXTRAPOLATE = False           # False: a target outside the calibrator range is a LIMIT; True: extrapolate (then 'uncertain')
EXTRAP_SLOPE_MIN, EXTRAP_SLOPE_MAX = 0.9, 2.5
# Bright targets (R CrB at 7-10 mag): the unsaturated calibrators stop at about 11-12 mag. With SAT_EXTEND the saturated
# calibrators (bloated images, flux still grows with image size) are added to the curve, only for a target brighter than the unsaturated range.
# The target is measured only if SAT_MIN_BRIGHTER calibrators are brighter than it (never beyond the brightest calibrator); the point is 'uncertain'.
SAT_EXTEND = True
SAT_MIN_CALIBRATORS = 4       # saturated-range calibrators needed to build the extended curve
SAT_MIN_BRIGHTER = 3          # calibrators that must be brighter than the target for it to count as inside the extended range
SAT_SCATTER_N = 8             # the error comes from the scatter of this many calibrators nearest the target in instrumental mag
SAT_SCATTER_FLOOR = 0.10      # smallest error added for a target in the extended range (mag)
# Edge of the calibrator range: a target up to EDGE_MAX_DIST (instrumental mag) brighter than the brightest calibrator is still read off the
# clamped curve (isophotal flux only). Checked against DASCH on R CrB, RY Sgr and UW Cen: scatter 0.5-0.7 mag, little bias. Beyond that it stays a limit.
EDGE_EXTEND = True
EDGE_MAX_DIST = 0.6
EDGE_ERR, EDGE_ERR_SLOPE = 0.7, 0.5   # error added in quadrature to an edge point: EDGE_ERR + EDGE_ERR_SLOPE * distance beyond the range (mag)
# Reference catalogue and its blend test (Gaia only). 'apass' keeps the old behaviour and the old output folder.
REF_CATALOG = 'gaia'          # 'gaia' or 'apass'
GAIA_MATCH_MAX_G = 18.0       # Gaia stars brighter than this are matched to the plate detections and used as calibrators
GAIA_ALL_MAX_G = 20.5         # Gaia stars down to this are used only for the blend test
GAIA_BLEND_ARCSEC = 10.0      # a reference star with a neighbour inside this radius ...
GAIA_BLEND_DMAG = 3.0         # ... that is less than this many mag fainter than it is not used as a calibrator
GAIA_MAX_PM = 40.0            # mas/yr; a faster star has moved too far between the plate epoch and 2016
GAIA_MAX_RUWE = 1.4           # worse astrometric fits are usually unresolved binaries
WCS_REF_MAX_V = 16.5          # Gaia stars brighter than this are used for the WCS offset search
OUT_SUBDIR = '02_A_gaia' if REF_CATALOG == 'gaia' else '02_A'
CURVE_PTS, CURVE_FRAC_LO = 20, 0.2
BRIGHT_N, BRIGHT_PTS, BRIGHT_FRAC_LO = 5, 10, 0.1
CAL_MAX_ELLIP = 0.8
TIER_FLAGS = ['f_trailed', 'f_poor_cal', 'f_pick_far', 'f_sat_ext', 'f_edge']
WCS_APPLY_MIN_PX, WCS_APPLY_MIN_STARS = 5.0, 20
LOCAL_R_PX, LOCAL_MIN_STARS = 150.0, 8
LOCAL_ZP_R_PX = 0.0           # local zero-point, off (it did not help)
LOCAL_ZP_MIN_STARS, LOCAL_ZP_SHRINK_N, LOCAL_ZP_FAINT_WIN = 8, 3.0, 2.5
DETECT_THRESH, DETECT_MINAREA = 1.5, 5
DEBLEND_NTHRESH, DEBLEND_MINCONT = 32, 0.005
SKY_MESH_PX = 128
SATUR_LEVEL = 32000.0
SEG_WINDOW_PX = 3
TARGET_SEARCH_PX = 25.0
MATCH_RADIUS_ARCSEC, MATCH_RADIUS_WIDE_ARCSEC = 10.0, 25.0
MIN_REF_STARS_GLOBAL, MIN_REF_STARS_ABSOLUTE_MIN = 25, 20
SIGNIF_THRESHOLD, OUTLIER_SIGMA = 5.0, 3.0
MAX_CALIBRATION_RMS = 2.5
COLOR_TERM_MIN_STARS, COLOR_TERM_SANITY_MAX = 30, 2.5
REF_BANDS = ['Bmag', 'Vmag']
EXTRAP_POINTS = 15
NEAR_N = 25
TRAIL_ELLIP, BIG_IMAGE_PX, POOR_CAL_RMS, PICK_FAR_PX = 0.5, 25.0, 0.8, 15.0
PIX_SCALE = 1.44
MIN_APASS_STARS = 50          # fewer APASS stars than this in the field: the star is skipped
APASS_SOURCES = ('gavo_dr10', 'vizier_dr9')   # tried in this order; ('vizier_dr9', 'gavo_dr10') to prefer DR9 (the CDS server does not answer on some networks)
PIPELINE_VERSION = 'sep-merge-3'
KINDS = ('iso', 'circ')

_SIG = hashlib.md5(repr((PIPELINE_VERSION, APER_R_PX, EXTRAPOLATE, CURVE_PTS, CURVE_FRAC_LO, BRIGHT_N, BRIGHT_PTS, BRIGHT_FRAC_LO, CAL_MAX_ELLIP,
                         DETECT_THRESH, DETECT_MINAREA, SKY_MESH_PX, MAX_CALIBRATION_RMS, LOCAL_ZP_R_PX,
                         SAT_EXTEND, SAT_MIN_CALIBRATORS, SAT_MIN_BRIGHTER, SAT_SCATTER_N, SAT_SCATTER_FLOOR)
                        + ((REF_CATALOG, GAIA_MATCH_MAX_G, GAIA_ALL_MAX_G, GAIA_BLEND_ARCSEC, GAIA_BLEND_DMAG, GAIA_MAX_PM, GAIA_MAX_RUWE, WCS_REF_MAX_V)
                           if REF_CATALOG == 'gaia' else ())).encode()).hexdigest()[:10]

# STAR FOLDERS, TARGET COORDINATES, PLATE LISTS, APASS FIELD
def sanitize_folder_name(name):
    cleaned = re.sub(r'[<>:"/\\|?*]', '', name).strip()
    return re.sub(r'\s+', '_', cleaned)

def exp_id_of(filename):
    stem = os.path.splitext(os.path.basename(filename))[0]
    return stem.split('_', 1)[1] if '_' in stem else stem

def read_targets_file():
    for folder in (BASE_DATA_DIR, BASE_DATA_DIR.parent / 'notebooks', BASE_DATA_DIR.parent, BASE_DATA_DIR.parent / 'project_Notes',
                   BASE_DATA_DIR.parent / 'test_CNN' / 'data'):
        f = folder / TARGETS_FILE_NAME
        if f.exists():
            out = {}
            for line in f.read_text(encoding='utf-8-sig').splitlines():
                s = line.strip()
                if not s or s.startswith('#'):
                    continue
                parts = [p.strip() for p in s.split('|')]
                if len(parts) == 3:
                    try:
                        out[sanitize_folder_name(parts[0])] = (float(parts[1]), float(parts[2]))
                    except ValueError:
                        pass
            return out
    return {}

TARGETS_FILE_COORDS = read_targets_file()
BASE_DATA_DIR.mkdir(parents=True, exist_ok=True)   # made here if it does not exist yet

def cutout_centre(star_dir, n=15):
    # Median sky position of the centre pixel over a sample of cutouts (the cutouts are centred on the target)
    files = sorted((star_dir / 'cutouts').glob('*.fits'))
    if not files:
        return None
    pick = files[::max(len(files) // n, 1)][:n]
    ras, decs = [], []
    for f in pick:
        try:
            h = fits.getheader(f)
            w = WCS(h)
            ra_, dec_ = w.all_pix2world((h['NAXIS1'] - 1) / 2.0, (h['NAXIS2'] - 1) / 2.0, 0)
            ras.append(float(ra_)); decs.append(float(dec_))
        except Exception:
            continue
    if not ras:
        return None
    return SkyCoord(ra=np.median(ras) * u.deg, dec=np.median(decs) * u.deg)

def resolve_target_coord(star_dir):
    name = star_dir.name
    if name in MANUAL_TARGET_COORDS:
        return MANUAL_TARGET_COORDS[name], 'manual'
    if name in TARGETS_FILE_COORDS:
        ra_, dec_ = TARGETS_FILE_COORDS[name]
        return SkyCoord(ra=ra_ * u.deg, dec=dec_ * u.deg), 'targets file'
    own = star_dir / 'target_coord.txt'   # written by the cutout notebook: the position the cutouts are centred on
    if own.exists():
        parts = own.read_text().split()
        if len(parts) >= 2:
            return SkyCoord(ra=float(parts[0]) * u.deg, dec=float(parts[1]) * u.deg), 'cutout notebook'
    tc = star_dir / '02_A' / 'target_coord.txt'
    if tc.exists():
        parts = tc.read_text().split()
        if len(parts) >= 3 and parts[2] == 'resolved':
            return SkyCoord(ra=float(parts[0]) * u.deg, dec=float(parts[1]) * u.deg), 'cached'
    try:
        coord = SkyCoord.from_name(name.replace('_', ' '))
        tc.parent.mkdir(parents=True, exist_ok=True)
        tc.write_text(f'{coord.ra.deg:.9f} {coord.dec.deg:.9f} resolved\n')
        return coord, 'resolved'
    except Exception:
        c = cutout_centre(star_dir)
        return (c, 'cutout centre') if c is not None else (None, 'FAILED')

def select_plates(star_dir):
    # Returns (list of cutout file names to process, text describing where the list came from)
    cut_dir = star_dir / 'cutouts'
    on_disk = {p.name: p for p in cut_dir.glob('*.fits')}
    if not ONLY_DASCH_ACCEPTED:
        return sorted(on_disk), 'all cutouts in the folder'
    manifest = star_dir / 'plate_manifest.csv'
    if manifest.exists():
        df = pd.read_csv(manifest)
        if 'dasch_accepted' in df.columns and 'filename' in df.columns:
            acc = df[(df['dasch_accepted'].astype(str).str.lower() == 'true') & (df['status'].astype(str) == 'downloaded')]
            names = {str(f).replace('\\', '/').split('/')[-1] for f in acc['filename'] if isinstance(f, str) and f}
            miss = len(names - set(on_disk))
            return sorted(n for n in names if n in on_disk), f'manifest dasch_accepted ({len(names)} listed, {miss} not on disk)'
    csv_ = star_dir / 'dasch_official_lightcurve.csv'
    if csv_.exists():
        d = pd.read_csv(csv_)
        d = d[(d['reject'] == 0) & d['magcal_magdep'].notna()]
        if 'exp_local_id' in d.columns:
            d = d[pd.to_numeric(d['exp_local_id'], errors='coerce').fillna(-1) >= 0]
        ids = {f"{str(s).strip()}{int(p):05d}m{int(m)}s{int(sl)}" for s, p, m, sl in zip(d['series'], d['platenum'], d['mosnum'], d['solnum'])}
        names = [n for n in on_disk if exp_id_of(n) in ids]
        return sorted(names), f'dasch_official_lightcurve.csv ({len(ids)} accepted rows)'
    return None, 'no plate_manifest.csv with a dasch_accepted column and no dasch_official_lightcurve.csv'

def fetch_text(url, timeout, tries):
    # Returns (text, None) or (None, text of the last error)
    last = None
    for attempt in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return resp.read().decode('utf-8', 'replace'), None
        except Exception as e:
            last = f'{type(e).__name__}: {e}'
            time.sleep(2 ** attempt)
    return None, last

def clean_apass(df):
    out = pd.DataFrame({c: pd.to_numeric(df[c], errors='coerce') for c in ['ra', 'dec', 'Bmag', 'Vmag']})
    ok = np.isfinite(out.values).all(axis=1) & (out['Bmag'] > 0) & (out['Bmag'] < 25) & (out['Vmag'] > 0) & (out['Vmag'] < 25)
    return out[ok].reset_index(drop=True)

def apass_from_gavo(ra_c, dec_c, radius_arcsec):
    # APASS DR10 from the GAVO data centre (Heidelberg), cone search, csv
    params = {'RA': ra_c, 'DEC': dec_c, 'SR': radius_arcsec / 3600.0, 'VERB': 2, 'MAXREC': 200000, 'RESPONSEFORMAT': 'csv'}
    txt, err = fetch_text('https://dc.g-vo.org/apass/q/cone/scs.xml?' + urllib.parse.urlencode(params), 90, 2)
    if txt is None:
        return None, err
    try:
        d = pd.read_csv(io.StringIO(txt), comment='#')
        return clean_apass(pd.DataFrame({'ra': d['ra'], 'dec': d['dec'], 'Bmag': d['mag_b'], 'Vmag': d['mag_v']})), None
    except Exception as e:
        return None, f'could not read the reply ({type(e).__name__}: {e}); it began: {txt[:150]!r}'

def parse_apass_tsv(text):
    rows = []
    for line in text.split('\n'):
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('-'):
            continue
        parts = line.split('\t')
        if len(parts) < 4:
            continue
        try:
            rows.append(tuple(float(p) for p in parts[:4]))
        except ValueError:
            continue
    return clean_apass(pd.DataFrame(rows, columns=['ra', 'dec', 'Bmag', 'Vmag']))

def apass_from_vizier(ra_c, dec_c, radius_arcsec):
    # APASS DR9 (II/336) from VizieR at CDS
    params = {'-source': 'II/336/apass9', '-c': f'{ra_c} {dec_c}', '-c.eq': 'J2000', '-c.rs': radius_arcsec,
              '-out': 'RAJ2000,DEJ2000,Bmag,Vmag', '-out.max': 200000}
    txt, err = fetch_text('https://vizier.cds.unistra.fr/viz-bin/asu-tsv?' + urllib.parse.urlencode(params), 25, 1)
    if txt is None:
        return None, err
    return parse_apass_tsv(txt), None

def gaia_from_gavo(ra_c, dec_c, radius_arcsec):
    # Gaia DR3 (light version) from the GAVO data centre, cone search, csv; returns the raw columns
    params = {'RA': ra_c, 'DEC': dec_c, 'SR': radius_arcsec / 3600.0, 'VERB': 2, 'MAXREC': 500000, 'RESPONSEFORMAT': 'csv'}
    txt, err = fetch_text('https://dc.g-vo.org/gaia/q3/cone/scs.xml?' + urllib.parse.urlencode(params), 240, 2)
    if txt is None:
        return None, err
    try:
        d = pd.read_csv(io.StringIO(txt), comment='#')
        d.columns = [str(c).strip().lower() for c in d.columns]
        out = pd.DataFrame({'ra': d['ra'], 'dec': d['dec'], 'g': d['phot_g_mean_mag'], 'bp': d['phot_bp_mean_mag'], 'rp': d['phot_rp_mean_mag']})
        for c in ('pmra', 'pmdec', 'ruwe'):
            out[c] = d[c] if c in d.columns else np.nan
        return out, None
    except Exception as e:
        return None, f'could not read the reply ({type(e).__name__}: {e}); it began: {txt[:150]!r}'

def gaia_to_bv(g, bp, rp):
    # Gaia DR3 G - V relation (Riello et al. 2021) for V; B from a rough linear B-V(BP-RP). A colour error in B only acts like a colour term, which is fitted anyway
    c = bp - rp
    v = g - (-0.02704 + 0.01424 * c - 0.2156 * c ** 2 + 0.01426 * c ** 3)
    b = v + 0.80 * c - 0.05
    ok = np.isfinite(c) & (c > -0.4) & (c < 2.8)
    return np.where(ok, b, np.nan), np.where(ok, v, np.nan)

def build_gaia_reference(raw):
    # Returns a table like the APASS one (ra, dec, Bmag, Vmag) plus iso_ok = usable as a calibrator (no close neighbour, slow, good fit, has a colour)
    d = raw.apply(pd.to_numeric, errors='coerce')
    d = d[d['g'].notna() & (d['g'] < GAIA_ALL_MAX_G) & d['ra'].notna() & d['dec'].notna()].reset_index(drop=True)
    n = len(d)
    ra0, dec0 = float(np.median(d['ra'])), float(np.median(d['dec']))
    x = (d['ra'].values - ra0) * math.cos(math.radians(dec0)) * 3600.0
    y = (d['dec'].values - dec0) * 3600.0
    g = d['g'].values
    blended = np.zeros(n, bool)
    pairs = cKDTree(np.c_[x, y]).query_pairs(GAIA_BLEND_ARCSEC, output_type='ndarray')
    if len(pairs):
        i, j = pairs[:, 0], pairs[:, 1]
        blended[i[g[j] < g[i] + GAIA_BLEND_DMAG]] = True      # a neighbour that is not much fainter spoils the star
        blended[j[g[i] < g[j] + GAIA_BLEND_DMAG]] = True
    b, v = gaia_to_bv(g, d['bp'].values, d['rp'].values)
    has_col = np.isfinite(b) & np.isfinite(v)
    pm = np.hypot(d['pmra'].values, d['pmdec'].values)
    slow = ~(pm > GAIA_MAX_PM)                                 # no proper motion (NaN) counts as slow
    good_fit = ~(d['ruwe'].values > GAIA_MAX_RUWE)
    out = pd.DataFrame({'ra': d['ra'].values, 'dec': d['dec'].values, 'Bmag': np.where(has_col, b, g), 'Vmag': np.where(has_col, v, g),
                        'iso_ok': has_col & ~blended & slow & good_fit})
    return out[g < GAIA_MATCH_MAX_G].reset_index(drop=True)    # fainter stars were used only for the blend test

def get_reference_field(star_dir, coord, out_dir):
    # Returns (table or None, text naming the source, list of error texts); the table has ra, dec, Bmag, Vmag (and iso_ok for Gaia)
    if REF_CATALOG != 'gaia':
        return get_apass_field(star_dir, coord, out_dir)
    path = out_dir / 'gaia_field_raw.csv'
    errors = []
    if path.exists() and path.stat().st_size > 0:
        raw = pd.read_csv(path)
    else:
        f = next(iter(sorted((star_dir / 'cutouts').glob('*.fits'))), None)
        size = max(fits.getheader(f)['NAXIS1'], fits.getheader(f)['NAXIS2']) if f is not None else 835
        radius = (0.7071 * size + 100) * PIX_SCALE
        raw, err = gaia_from_gavo(coord.ra.deg, coord.dec.deg, radius)
        if raw is None or not len(raw):
            return None, '', [f'Gaia DR3 (GAVO): {err or "answered, but with no stars"}']
        raw.to_csv(path, index=False)
    ref = build_gaia_reference(raw)
    return ref, f'Gaia DR3 (GAVO): {len(raw)} stars, {len(ref)} to G<{GAIA_MATCH_MAX_G:g}, {int(ref["iso_ok"].sum())} usable as calibrators', errors

def get_apass_field(star_dir, coord, out_dir):
    # Returns (table or None, name of the source used, list of error texts)
    path, src_path = out_dir / 'apass_field.csv', out_dir / 'apass_source.txt'
    if path.exists():
        df = pd.read_csv(path)
        if len(df):
            return df, (src_path.read_text().strip() if src_path.exists() else 'saved apass_field.csv'), []
    f = next(iter(sorted((star_dir / 'cutouts').glob('*.fits'))), None)
    size = max(fits.getheader(f)['NAXIS1'], fits.getheader(f)['NAXIS2']) if f is not None else 835
    radius = (0.7071 * size + 100) * PIX_SCALE      # half diagonal plus room for a plate offset
    errors = []
    for key in APASS_SOURCES:
        name, fn = {'gavo_dr10': ('APASS DR10 (GAVO)', apass_from_gavo), 'vizier_dr9': ('APASS DR9 (VizieR, CDS)', apass_from_vizier)}[key]
        df, err = fn(coord.ra.deg, coord.dec.deg, radius)
        if df is not None and len(df):
            df.to_csv(path, index=False)
            src_path.write_text(name + '\n')
            return df, name, errors
        errors.append(f'{name}: {err or "answered, but with no stars"}')
    return None, '', errors

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess as _sm_lowess
    def lowess_fit(x, y, frac=0.3):
        out = _sm_lowess(y, x, frac=frac, return_sorted=True)
        return out[:, 0], out[:, 1]
except ImportError:
    def lowess_fit(x, y, frac=0.3):
        order = np.argsort(x)
        xs, ys = x[order], y[order]
        n = len(xs)
        span = max(int(frac * n), 4)
        fitted = np.empty(n)
        for i in range(n):
            lo = max(0, min(i - span // 2, n - span))
            hi = min(n, lo + span)
            xw, yw = xs[lo:hi], ys[lo:hi]
            if len(xw) < 2 or np.ptp(xw) == 0:
                fitted[i] = np.median(yw)
            else:
                slope, intercept, *_ = stats.linregress(xw, yw)
                fitted[i] = slope * xs[i] + intercept
        return xs, fitted



# PHOTOMETRY CORE (V CrA pipeline: sep detection, WCS repair, adaptive LOWESS calibration)
# Set per star by setup_star()
apass = None
apass_coords = None
TARGET_RA_DEG = TARGET_DEC_DEG = None

apass_wcs = None

def setup_star(apass_df, coord):
    global apass, apass_coords, apass_wcs, TARGET_RA_DEG, TARGET_DEC_DEG
    apass = apass_df.dropna(subset=['Bmag', 'Vmag']).reset_index(drop=True).copy()
    apass['color'] = apass['Bmag'] - apass['Vmag']
    if 'iso_ok' not in apass.columns:
        apass['iso_ok'] = True
    apass['iso_ok'] = apass['iso_ok'].astype(bool)
    apass_wcs = apass
    if REF_CATALOG == 'gaia' and (apass['Vmag'] <= WCS_REF_MAX_V).sum() >= 30:
        apass_wcs = apass[apass['Vmag'] <= WCS_REF_MAX_V]     # a deep catalogue would only add chance matches to the offset search
    apass_coords = SkyCoord(ra=apass['ra'].values * u.deg, dec=apass['dec'].values * u.deg)
    TARGET_RA_DEG, TARGET_DEC_DEG = float(coord.ra.deg), float(coord.dec.deg)

def measure_all(data):
    # One detection run; returns the same detections with isophotal flux ('iso') and with a circle of APER_R_PX ('circ')
    data = data.astype(np.float64)
    bkg = sep.Background(data, bw=min(SKY_MESH_PX, data.shape[1]), bh=min(SKY_MESH_PX, data.shape[0]))
    data_sub = data - bkg.back()
    rms = bkg.globalrms
    obj, segmap = sep.extract(
        data_sub, DETECT_THRESH, err=rms, minarea=DETECT_MINAREA,
        deblend_nthresh=DEBLEND_NTHRESH, deblend_cont=DEBLEND_MINCONT,
        clean=False, segmentation_map=True)
    if len(obj) == 0:
        return None
    h, w = data.shape
    a = np.maximum(obj['a'], 0.5)
    b = np.minimum(np.maximum(obj['b'], 0.5), a)
    iy = np.clip(np.round(obj['y']).astype(int), 0, h - 1)
    ix = np.clip(np.round(obj['x']).astype(int), 0, w - 1)
    t = pd.DataFrame({'x': obj['x'], 'y': obj['y'], 'ellipticity': 1.0 - b / a,
                      'peak': data[iy, ix], 'npix': obj['npix'], 'sigma_px': np.sqrt(a * b)})
    t['saturated'] = t['peak'] >= SATUR_LEVEL
    t['r_iso_px'] = np.sqrt(obj['npix'] / math.pi)
    t['flag'] = 0
    tabs = {}
    for kind in KINDS:
        tk = t.copy()
        if kind == 'iso':
            tk['flux'] = obj['flux']
            tk['fluxerr'] = np.sqrt(np.abs(obj['flux']) + obj['npix'] * rms ** 2)
        else:
            f_, fe_, fl_ = sep.sum_circle(data_sub, obj['x'], obj['y'], APER_R_PX, err=rms, gain=1.0, subpix=5)
            tk['flux'], tk['fluxerr'] = f_, fe_
        with np.errstate(divide='ignore', invalid='ignore'):
            tk['snr'] = tk['flux'] / tk['fluxerr']
            tk['mag_inst'] = -2.5 * np.log10(tk['flux'])
            tk['mag_inst_err'] = 1.0857 * tk['fluxerr'] / tk['flux']
        tabs[kind] = tk
    return {'tabs': tabs, 'segmap': segmap}


def pick_seg(segmap, tx, ty):
    # Object index whose segmentation covers the position (most common id in a small window)
    h, w = segmap.shape
    ix, iy = int(round(tx)), int(round(ty))
    win = segmap[max(iy - SEG_WINDOW_PX, 0):min(iy + SEG_WINDOW_PX + 1, h),
                 max(ix - SEG_WINDOW_PX, 0):min(ix + SEG_WINDOW_PX + 1, w)]
    ids = win[win > 0]
    if len(ids) == 0:
        return None
    vals, cnt = np.unique(ids, return_counts=True)
    return int(vals[np.argmax(cnt)]) - 1

# WCS repair: find the offset between detections and APASS

def shift_search(tbl, cx, cy, max_px, step, tol, fine, n_det):
    res = {'found': False, 'dx': 0.0, 'dy': 0.0, 'n0': 0, 'n': 0}
    t = tbl[tbl['flux'] > 0].sort_values('flux', ascending=False).head(n_det)
    if len(t) < 20 or len(cx) < 20:
        return res
    px, py = t['x'].values, t['y'].values
    tree = cKDTree(np.c_[cx, cy])
    grid = np.arange(-max_px, max_px + 0.1, step)
    counts = np.zeros((len(grid), len(grid)), int)
    for i, sx in enumerate(grid):
        for j, sy in enumerate(grid):
            d, _ = tree.query(np.c_[px - sx, py - sy], distance_upper_bound=tol)
            counts[i, j] = np.isfinite(d).sum()
    iz = len(grid) // 2
    res['n0'] = int(counts[iz, iz])
    base, best = float(np.median(counts)), counts.max()
    if best < base + 5 * math.sqrt(max(base, 1.0)) + 5:
        return res
    cand = np.argwhere(counts == best)
    k = cand[np.argmin(np.hypot(grid[cand[:, 0]], grid[cand[:, 1]]))]
    sx, sy = grid[k[0]], grid[k[1]]
    d, ii = tree.query(np.c_[px - sx, py - sy], distance_upper_bound=fine)
    sel = np.isfinite(d)
    ox, oy = px[sel] - cx[ii[sel]], py[sel] - cy[ii[sel]]
    keep = np.ones(len(ox), bool)
    for _ in range(5):
        if keep.sum() < 5:
            break
        mx, my = np.median(ox[keep]), np.median(oy[keep])
        sdx, sdy = max(np.std(ox[keep]), 0.3), max(np.std(oy[keep]), 0.3)
        new = (np.abs(ox - mx) < 3 * sdx) & (np.abs(oy - my) < 3 * sdy)
        if new.sum() < 5 or (new == keep).all():
            break
        keep = new
    res.update({'found': True, 'dx': float(np.median(ox[keep])), 'dy': float(np.median(oy[keep])), 'n': int(keep.sum()),
                'pairs': (px[sel][keep], py[sel][keep], ox[keep], oy[keep])})
    return res

def find_wcs_shift(tbl, cx, cy):
    # Stage 1: +/-40 px. Stage 2 (only if stage 1 finds nothing): +/-100 px, coarser grid, more detections
    r = shift_search(tbl, cx, cy, 40, 2.0, 3.0, 4.0, 300)
    r['stage'] = 1 if r['found'] else 0
    if not r['found']:
        r2 = shift_search(tbl, cx, cy, 100, 4.0, 5.0, 6.0, 500)
        if r2['found']:
            r2['stage'] = 2
            r2['n0'] = r['n0']
            return r2
    return r

# Calibration

def bisector_fit(x, y):
    xm, ym = np.mean(x), np.mean(y)
    Sxx, Syy, Sxy = np.sum((x - xm) ** 2), np.sum((y - ym) ** 2), np.sum((x - xm) * (y - ym))
    if Sxy == 0 or Sxx == 0:
        return 0.0
    b1, b2 = Sxy / Sxx, Syy / Sxy
    return (b1 * b2 - 1 + ((1 + b1 ** 2) * (1 + b2 ** 2)) ** 0.5) / (b1 + b2)

def adaptive_frac(n, pts=None, frac_lo=None):
    pts = CURVE_PTS if pts is None else pts
    frac_lo = CURVE_FRAC_LO if frac_lo is None else frac_lo
    return float(np.clip(pts / max(n, 1), frac_lo, 0.6))

def global_calibration(inst_mag, ref_mag, color, pts=None, frac_lo=None):
    order = np.argsort(inst_mag)
    m, r, c = inst_mag[order], ref_mag[order], color[order]
    xs, fitted = lowess_fit(m, r, frac=adaptive_frac(len(m), pts, frac_lo))
    resid = r - np.interp(m, xs, fitted)
    for sigma in (OUTLIER_SIGMA, OUTLIER_SIGMA, 4.0):
        keep = np.abs(resid) < sigma * np.std(resid)
        if keep.sum() < MIN_REF_STARS_ABSOLUTE_MIN:
            break
        m, r, c = m[keep], r[keep], c[keep]
        xs, fitted = lowess_fit(m, r, frac=adaptive_frac(len(m), pts, frac_lo))
        resid = r - np.interp(m, xs, fitted)
    cterm = 0.0
    if len(m) >= COLOR_TERM_MIN_STARS:
        trial = bisector_fit(c, resid)
        if abs(trial) <= COLOR_TERM_SANITY_MAX:
            rc = r - c * trial
            xs2, f2 = lowess_fit(m, rc, frac=adaptive_frac(len(m), pts, frac_lo))
            res2 = rc - np.interp(m, xs2, f2)
            if np.std(res2) < np.std(resid):
                cterm, xs, fitted, resid = trial, xs2, f2, res2
    return {'inst_mag': m, 'resid': resid, 'curve_x': xs, 'curve_y': fitted,
            'colorterm': cterm, 'rms': float(np.std(resid)), 'n': len(m)}

def curve_mag(cal, m, extrapolate):
    # Clamped at the curve ends, or extended from the curve end with a clipped slope
    cx, cy = cal['curve_x'], cal['curve_y']
    if cx.min() <= m <= cx.max() or not extrapolate:
        return float(np.interp(m, cx, cy))
    k = min(EXTRAP_POINTS, len(cx))
    if m < cx.min():
        slope, x_end, y_end = np.polyfit(cx[:k], cy[:k], 1)[0], cx[0], cy[0]
    else:
        slope, x_end, y_end = np.polyfit(cx[-k:], cy[-k:], 1)[0], cx[-1], cy[-1]
    if not np.isfinite(slope):
        return float(np.interp(m, cx, cy))
    slope = float(np.clip(slope, EXTRAP_SLOPE_MIN, EXTRAP_SLOPE_MAX))
    return float(y_end + slope * (m - x_end))

def local_scatter(cal, m_target):
    # Robust scatter of the NEAR_N calibrators closest to the target in instrumental mag
    idx = np.argsort(np.abs(cal['inst_mag'] - m_target))[:NEAR_N]
    r = cal['resid'][idx]
    if len(r) < 8:
        return cal['rms']
    return max(1.4826 * float(np.median(np.abs(r - np.median(r)))), 0.02)

def calibrate_target(t, wcs, row, shift=(0.0, 0.0)):
    # t holds the calibrator detections with x, y already moved into the catalog frame
    r = {'status': 'ok'}
    ra, dec = wcs.all_pix2world(t['x'].values, t['y'].values, 0)
    idx, sep2d, _ = SkyCoord(ra=ra * u.deg, dec=dec * u.deg).match_to_catalog_sky(apass_coords)
    matched = sep2d.arcsec < MATCH_RADIUS_ARCSEC
    if matched.sum() < MIN_REF_STARS_GLOBAL:
        matched = sep2d.arcsec < MATCH_RADIUS_WIDE_ARCSEC
    ref_df = apass.iloc[idx[matched]].reset_index(drop=True)
    det_df = t[matched].reset_index(drop=True)
    good_all = (det_df['flag'] == 0) & (det_df['snr'] > SIGNIF_THRESHOLD) & (det_df['ellipticity'] < CAL_MAX_ELLIP) & ref_df['iso_ok']
    good = good_all & (~det_df['saturated'])
    ref_all, det_all = ref_df[good_all].reset_index(drop=True), det_df[good_all].reset_index(drop=True)   # saturated calibrators kept, for SAT_EXTEND
    ref_df, det_df = ref_df[good].reset_index(drop=True), det_df[good].reset_index(drop=True)
    if len(det_df) < MIN_REF_STARS_ABSOLUTE_MIN:
        r['status'] = 'too few matched ref stars'
        return r
    cal = None
    for band in REF_BANDS:
        trial = global_calibration(det_df['mag_inst'].values, ref_df[band].values, ref_df['color'].values)
        if cal is None or trial['rms'] < cal['rms']:
            cal = trial
            cal['band'] = band
    r.update({'cal_rms': cal['rms'], 'n_ref': cal['n'], 'ref_band': cal['band'], 'colorterm': cal['colorterm']})
    if cal['rms'] > MAX_CALIBRATION_RMS:
        r['status'] = 'calibration RMS too high'
        return r
    # Plate depth: calibrated mag where the 95th percentile faintest SNR>5 detection sits (clamped at the curve end)
    s_ = t.loc[t['snr'] > SIGNIF_THRESHOLD, 'mag_inst']
    if len(s_) >= 10:
        r['plate_limit'] = float(np.interp(np.percentile(s_, 95), cal['curve_x'], cal['curve_y']))
    if row['snr'] < SIGNIF_THRESHOLD:
        r['status'] = 'target below SNR threshold'
        r['limiting_mag'] = float(np.percentile(cal['curve_y'], 95))
        return r

    # Magnitude: clamped at the curve ends (a limit if outside), or extrapolated if EXTRAPOLATE
    m = float(row['mag_inst'])
    lo, hi = cal['curve_x'].min(), cal['curve_x'].max()
    r['in_range'] = bool(lo <= m <= hi)
    r['limit_dir'] = '' if r['in_range'] else ('brighter' if m < lo else 'fainter')
    r['extrap_dist'] = float(max(lo - m, m - hi, 0.0))
    r['n_ref_brighter'] = int((cal['inst_mag'] < m).sum())
    kb = min(EXTRAP_POINTS, len(cal['curve_x']))
    r['bright_slope'] = float(np.polyfit(cal['curve_x'][:kb], cal['curve_y'][:kb], 1)[0])
    # Target near the bright end (few calibrators brighter): read the magnitude off a less smoothed curve
    cal_m = cal
    r['bright_curve'] = False
    r['bright_err'] = 0.0
    if BRIGHT_N > 0 and r['n_ref_brighter'] <= BRIGHT_N:
        c2 = global_calibration(det_df['mag_inst'].values, ref_df[cal['band']].values, ref_df['color'].values, BRIGHT_PTS, BRIGHT_FRAC_LO)
        if np.isfinite(c2['rms']) and len(c2['curve_x']) > 5:
            cal_m = c2
            r['bright_curve'] = True
            r['bright_err'] = 0.5 * abs(curve_mag(c2, m, False) - curve_mag(cal, m, False))
    # Target brighter than the unsaturated calibrators: add the saturated calibrators to the curve
    r['sat_ext'] = False
    r['sat_ext_n'] = 0
    if SAT_EXTEND and m < lo and len(det_all) > len(det_df):
        n_bright = int((det_all['mag_inst'] < lo).sum())
        if n_bright >= SAT_MIN_CALIBRATORS:
            cal_s = global_calibration(det_all['mag_inst'].values, ref_all[cal['band']].values, ref_all['color'].values, BRIGHT_PTS, BRIGHT_FRAC_LO)
            if np.isfinite(cal_s['rms']) and len(cal_s['curve_x']) > 5:
                n_brighter = int((cal_s['inst_mag'] < m).sum())
                cal_m = cal_s
                r['bright_curve'] = True
                r['sat_ext_n'] = n_brighter
                r['bright_slope'] = float(np.polyfit(cal_s['curve_x'][:kb], cal_s['curve_y'][:kb], 1)[0])
                lo_s = cal_s['curve_x'].min()
                if n_brighter >= SAT_MIN_BRIGHTER:
                    r['sat_ext'] = True
                    r['in_range'], r['limit_dir'], r['extrap_dist'] = True, '', 0.0
                    r['n_ref_brighter'] = n_brighter
                    # Honest error: leave each of the nearest calibrators out, refit, and see how well the curve predicts it
                    inst_a, ref_a, col_a = det_all['mag_inst'].values, ref_all[cal['band']].values, ref_all['color'].values
                    loo = []
                    for j in np.argsort(np.abs(inst_a - m))[:SAT_SCATTER_N]:
                        keep_ = np.arange(len(inst_a)) != j
                        c_j = global_calibration(inst_a[keep_], ref_a[keep_], col_a[keep_], BRIGHT_PTS, BRIGHT_FRAC_LO)
                        if np.isfinite(c_j['rms']) and len(c_j['curve_x']) > 5:
                            loo.append(ref_a[j] - col_a[j] * c_j['colorterm'] - curve_mag(c_j, inst_a[j], False))
                    r['bright_err'] = max(float(np.sqrt(np.mean(np.square(loo)))) if len(loo) >= 3 else float(np.std(cal_s['resid'])), SAT_SCATTER_FLOOR)
                else:
                    r['extrap_dist'] = float(max(lo_s - m, 0.0))   # still brighter than everything: a limit, now at the brightest calibrator of all
    r['mag_clamped'] = curve_mag(cal_m, m, False)
    r['mag_extrap'] = curve_mag(cal_m, m, True)
    # Local zero-point: median residual of the calibrators near the target, relative to the curve used for the target
    r['zp_local'], r['zp_n'] = 0.0, 0
    if LOCAL_ZP_R_PX > 0:
        tx, ty = float(row['x']) - shift[0], float(row['y']) - shift[1]
        near = (np.hypot(det_df['x'].values - tx, det_df['y'].values - ty) < LOCAL_ZP_R_PX) & (det_df['mag_inst'].values < m + LOCAL_ZP_FAINT_WIN)
        if near.sum() >= LOCAL_ZP_MIN_STARS:
            mi = det_df['mag_inst'].values[near]
            ri = ref_df[cal['band']].values[near] - ref_df['color'].values[near] * cal_m['colorterm']
            zr = ri - np.array([curve_mag(cal_m, x_, False) for x_ in mi])
            zr = zr[np.abs(zr - np.median(zr)) < 3 * max(1.4826 * np.median(np.abs(zr - np.median(zr))), 0.03)]
            if len(zr) >= LOCAL_ZP_MIN_STARS:
                zp = float(np.median(zr)) * len(zr) / (len(zr) + LOCAL_ZP_SHRINK_N)
                r['zp_local'], r['zp_n'] = zp, int(len(zr))
                r['mag_clamped'] += zp
                r['mag_extrap'] += zp
    r['mag'] = r['mag_extrap'] if EXTRAPOLATE else r['mag_clamped']
    r['scatter_local'] = local_scatter(cal, m)
    r['extrap_err'] = 0.5 * abs(r['mag_extrap'] - r['mag_clamped']) if EXTRAPOLATE else 0.0
    r['mag_err'] = float(math.sqrt(row['mag_inst_err'] ** 2 + r['scatter_local'] ** 2 + r['extrap_err'] ** 2 + r['bright_err'] ** 2))
    return r

# Pick the target near the expected position, calibrate, measure

def evaluate(tbl, segmap, wcs, tpos, shift):
    out = {'status': 'ok'}
    k = pick_seg(segmap, tpos[0], tpos[1])
    if k is None:
        d = np.hypot(tbl['x'].values - tpos[0], tbl['y'].values - tpos[1])
        if d.min() > TARGET_SEARCH_PX:
            out['status'] = 'no object near target'
            return out
        k = int(np.argmin(d))
    row = tbl.iloc[k]
    out.update({'x': float(row['x']), 'y': float(row['y']),
                'pick_dist_px': float(math.hypot(row['x'] - tpos[0], row['y'] - tpos[1])),
                'pick_r_iso_px': float(row['r_iso_px']), 'pick_ellipticity': float(row['ellipticity']),
                'saturated': bool(row['saturated'])})
    if not (row['flux'] > 0 and row['fluxerr'] > 0):
        out['status'] = 'target flux non-positive'
        return out
    out['mag_inst'], out['snr'] = float(row['mag_inst']), float(row['snr'])
    t = tbl[(tbl['flux'] > 0) & (np.arange(len(tbl)) != k)].reset_index(drop=True)
    if len(t) < MIN_REF_STARS_ABSOLUTE_MIN:
        out['status'] = 'too few detections'
        return out
    t = t.assign(x=t['x'].values - shift[0], y=t['y'].values - shift[1])
    out.update(calibrate_target(t, wcs, row, shift))
    return out


# ONE PLATE (both flux kinds from one detection run)
def plate_date_jd(header):
    for key in ('DATE-OBS', 'DATE-AVG'):
        if key in header:
            try:
                return float(Time(header[key]).jd)
            except Exception:
                pass
    if 'JD-OBS' in header:
        try:
            return float(header['JD-OBS'])
        except Exception:
            pass
    if 'MJD-OBS' in header:
        try:
            return float(header['MJD-OBS']) + 2400000.5
        except Exception:
            pass
    return np.nan

def solve_kind(tbl, segmap, wcs, tx, ty, w, h):
    # WCS offset search, three versions of the measurement, the repair rule; returns the fields of one result row
    out = {}
    sh = {'found': False, 'dx': 0.0, 'dy': 0.0, 'n0': 0, 'n': 0, 'stage': 0}
    cx, cy = wcs.all_world2pix(apass_wcs['ra'].values, apass_wcs['dec'].values, 0)
    inb = (cx > -100) & (cx < w + 100) & (cy > -100) & (cy < h + 100)
    sh = find_wcs_shift(tbl, cx[inb], cy[inb])
    cand = bool(sh['found'] and math.hypot(sh['dx'], sh['dy']) >= WCS_APPLY_MIN_PX and sh['n'] >= WCS_APPLY_MIN_STARS)
    ldx, ldy, nloc = sh['dx'], sh['dy'], 0
    if cand:
        near = np.hypot(sh['pairs'][0] - (tx + sh['dx']), sh['pairs'][1] - (ty + sh['dy'])) < LOCAL_R_PX
        nloc = int(near.sum())
        if nloc >= LOCAL_MIN_STARS:
            ldx, ldy = float(np.median(sh['pairs'][2][near])), float(np.median(sh['pairs'][3][near]))
    out.update({'wcs_found': bool(sh['found']), 'wcs_stage': sh['stage'], 'wcs_dx': sh['dx'], 'wcs_dy': sh['dy'],
                'wcs_n': sh['n'], 'wcs_n0': sh['n0'], 'wcs_cand': cand,
                'wcs_local_dx': ldx, 'wcs_local_dy': ldy, 'wcs_n_local': nloc})
    base = evaluate(tbl, segmap, wcs, (tx, ty), (0.0, 0.0))
    chosen, choice = base, 'none'
    out.update({'mag_nowcs': base.get('mag', np.nan), 'status_nowcs': base['status'], 'cal_rms_nowcs': base.get('cal_rms', np.nan),
                'mag_global': np.nan, 'mag_local': np.nan})
    if cand:
        glob = evaluate(tbl, segmap, wcs, (tx + sh['dx'], ty + sh['dy']), (sh['dx'], sh['dy']))
        if nloc >= LOCAL_MIN_STARS and math.hypot(ldx - sh['dx'], ldy - sh['dy']) > 1.0:
            loc = evaluate(tbl, segmap, wcs, (tx + ldx, ty + ldy), (sh['dx'], sh['dy']))
        else:
            loc = glob
        out.update({'mag_global': glob.get('mag', np.nan), 'mag_local': loc.get('mag', np.nan)})
        rep_ = loc if loc['status'] == 'ok' else glob
        # Repaired version only if its calibration is tighter; if it gives a limit or a far pick while the unrepaired one is a clean pick, keep the unrepaired
        if rep_['status'] == 'ok' and (base['status'] != 'ok' or rep_.get('cal_rms', 9.0) < base.get('cal_rms', 9.0)):
            chosen, choice = rep_, 'repaired'
            rep_bad = (rep_.get('in_range', True) is False) or (rep_.get('pick_dist_px', 0.0) > PICK_FAR_PX)
            base_ok = base['status'] == 'ok' and (base.get('in_range', False) is True) and (base.get('pick_dist_px', 99.0) <= PICK_FAR_PX)
            if rep_bad and base_ok:
                chosen, choice = base, 'fallback'
        else:
            choice = 'kept_base'
    out['wcs_choice'] = choice
    out['wcs_applied'] = choice == 'repaired'
    out.update(chosen)
    out['cal_rms_final'] = chosen.get('cal_rms', np.nan)
    return out

def process_plate(path):
    # Returns {'iso': row, 'circ': row}; both rows carry file, exp_id, date_jd, status
    name = os.path.basename(path)
    base = {'file': name, 'exp_id': exp_id_of(name), 'status': 'ok'}
    with fits.open(path) as hd:
        data, header = hd[0].data, hd[0].header
        wcs = WCS(header)
    if not wcs.has_celestial:
        return {k: {**base, 'status': 'no WCS'} for k in KINDS}
    base['date_jd'] = plate_date_jd(header)
    tx, ty = [float(v) for v in wcs.all_world2pix(TARGET_RA_DEG, TARGET_DEC_DEG, 0)]
    h, w = data.shape
    if not (0 <= tx < w and 0 <= ty < h):
        return {k: {**base, 'status': 'target position outside cutout'} for k in KINDS}
    M = measure_all(data)
    if M is None:
        return {k: {**base, 'status': 'no detections'} for k in KINDS}
    res = {}
    for kind in KINDS:
        tbl = M['tabs'][kind]
        out = dict(base)
        out['n_det'] = len(tbl)
        st = tbl[(~tbl['saturated']) & (tbl['ellipticity'] < 0.3) & (tbl['npix'] >= 8)]
        s1 = st[(st['snr'] > 15) & (st['snr'] < 60)]
        if len(s1) < 10:
            s1 = st[st['snr'] > 15]
        out['fwhm_arcsec'] = 2.355 * float(np.median(s1['sigma_px'])) * PIX_SCALE if len(s1) >= 10 else np.nan
        out.update(solve_kind(tbl, M['segmap'], wcs, tx, ty, w, h))
        res[kind] = out
    return res

# FLAGS, TIERS AND THE MERGE
NUM_COLS = ['mag', 'mag_err', 'in_range', 'extrap_dist', 'n_ref_brighter', 'pick_dist_px', 'pick_r_iso_px', 'pick_ellipticity',
            'cal_rms', 'date_jd', 'limit_dir', 'bright_slope', 'wcs_found', 'wcs_applied', 'wcs_dx', 'wcs_dy', 'wcs_n', 'wcs_n0',
            'wcs_stage', 'mag_nowcs', 'n_det', 'fwhm_arcsec', 'plate_limit', 'wcs_cand', 'wcs_choice', 'mag_global', 'mag_local',
            'wcs_n_local', 'wcs_local_dx', 'wcs_local_dy', 'cal_rms_nowcs', 'bright_curve', 'zp_local', 'zp_n', 'snr', 'mag_inst', 'sat_ext', 'sat_ext_n']

def tier_frame(rows, manual, edge_ok=False):
    res = pd.DataFrame(rows)
    for c in NUM_COLS:
        if c not in res.columns:
            res[c] = np.nan
    res['manual_reject'] = res['file'].isin(manual)
    # Edge of the range (isophotal table only): just brighter than the brightest calibrator, kept with a larger error
    res['f_edge'] = False
    if EDGE_EXTEND and edge_ok:
        res['mag_err'] = pd.to_numeric(res['mag_err'], errors='coerce')
        res['f_edge'] = ((res['in_range'] == False) & (res['limit_dir'] == 'brighter') & (pd.to_numeric(res['extrap_dist'], errors='coerce') <= EDGE_MAX_DIST)
                         & pd.to_numeric(res['mag'], errors='coerce').notna() & (res['status'] == 'ok'))
        res.loc[res['f_edge'], 'mag_err'] = np.sqrt(res.loc[res['f_edge'], 'mag_err'] ** 2 + (EDGE_ERR + EDGE_ERR_SLOPE * pd.to_numeric(res.loc[res['f_edge'], 'extrap_dist'], errors='coerce')) ** 2)
    res['f_extrap'] = (res['in_range'] == False) & ~res['f_edge']
    res['f_trailed'] = res['pick_ellipticity'] > TRAIL_ELLIP
    res['f_big_image'] = res['pick_r_iso_px'] > BIG_IMAGE_PX
    res['f_poor_cal'] = res['cal_rms'] > POOR_CAL_RMS
    res['f_pick_far'] = res['pick_dist_px'] > PICK_FAR_PX
    res['f_wcs_applied'] = res['wcs_applied'] == True
    res['f_wcs_unresolved'] = res['wcs_found'] == False
    res['f_sat_ext'] = res['sat_ext'] == True
    flag_cols = ['f_extrap', 'f_trailed', 'f_big_image', 'f_poor_cal', 'f_pick_far', 'f_sat_ext', 'f_edge', 'f_wcs_applied', 'f_wcs_unresolved']
    res['flags'] = res[flag_cols].apply(lambda r: ','.join(c[2:] for c in flag_cols if bool(r[c])), axis=1)
    other = [c for c in TIER_FLAGS if c != 'f_pick_far'] + (['f_extrap'] if EXTRAPOLATE else [])
    res['tier'] = np.select(
        [res['manual_reject'] | (res['status'] != 'ok') | res['f_pick_far'],
         res['f_extrap'] & (not EXTRAPOLATE) & res['f_poor_cal'],
         res['f_extrap'] & (not EXTRAPOLATE),
         res[other].any(axis=1)],
        ['reject', 'limit_weak', 'limit', 'uncertain'], default='ok')
    return res.set_index('file', drop=False)

PUB = ('ok', 'uncertain')

def merge_kinds(A, B):
    # A = isophotal table, B = circle table (same index). Rule uses only our own numbers
    A = A.reindex(columns=sorted(set(A.columns) | set(B.columns)))
    B = B.reindex(columns=A.columns)
    a_pub = A['tier'].isin(PUB) & A['mag'].notna()
    slope = pd.to_numeric(B['bright_slope'], errors='coerce')
    b_pub = B['tier'].isin(PUB) & B['mag'].notna() & (slope >= BAD_SLOPE)
    b_limit = B['tier'] == 'limit'
    use_iso = (b_pub & (B['mag'] >= ISO_FROM_MAG) & a_pub) | (~b_pub & a_pub & (B['tier'] != 'reject')) | (FALLBACK_LIMITS & b_limit & a_pub)
    # Target brighter than the unsaturated calibrators (iso needed the saturated ones): its image is bigger than the circle, so the circle is cut off
    use_iso = use_iso | ((A['sat_ext'] == True) & a_pub)
    # Edge of the range: the isophotal value, unless the circle is a clean measurement
    use_iso = use_iso | ((A['f_edge'] == True) & a_pub & ~b_pub)
    m = B.where(~use_iso, A, axis=0).copy()
    m['source'] = np.where(use_iso, 'iso', 'circ')
    bad_slope_only = (slope < BAD_SLOPE) & ~use_iso & B['tier'].isin(PUB)
    m.loc[bad_slope_only, 'tier'] = 'reject'
    m['mag_iso'], m['tier_iso'] = A['mag'], A['tier']
    m['mag_circ'], m['tier_circ'] = B['mag'], B['tier']
    for c in ['mag', 'mag_err', 'date_jd', 'bright_slope', 'cal_rms', 'pick_dist_px', 'snr', 'pick_ellipticity']:
        m[c] = pd.to_numeric(m[c], errors='coerce')
    m['manual_reject'] = m['manual_reject'].astype(str).str.lower() == 'true'
    return m

# PER-STAR RUNNER
def format_eta(seconds):
    if seconds is None or seconds != seconds or seconds < 0:
        return 'calculating...'
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f'{h:d}:{m:02d}:{s:02d}' if h else f'{m:d}:{s:02d}'

def run_star(star_dir, coord, coord_source, bar, html, label):
    out_dir = star_dir / OUT_SUBDIR
    out_dir.mkdir(parents=True, exist_ok=True)
    cache_dir = out_dir / 'algorithm_cache_sep'
    cache_dir.mkdir(parents=True, exist_ok=True)

    plates, plate_src = select_plates(star_dir)
    if plates is None:
        raise RuntimeError(f'cannot tell which plates DASCH accepted: {plate_src}. Run dasch_Cutout_Recreation for this star, or set ONLY_DASCH_ACCEPTED = False')
    if PLATE_LIMIT_PER_STAR:
        plates = plates[:PLATE_LIMIT_PER_STAR]
    if not plates:
        raise RuntimeError(f'no plates to process ({plate_src})')

    centre = cutout_centre(star_dir)
    off = coord.separation(centre).arcsec if centre is not None else np.nan
    apass_df, apass_src, apass_errs = get_reference_field(star_dir, coord, out_dir)
    if apass_df is None:
        raise RuntimeError('no reference catalogue answered. ' + ' | '.join(apass_errs))
    if len(apass_df) < MIN_APASS_STARS:
        raise RuntimeError(f'only {len(apass_df)} reference stars with B and V in the field (need {MIN_APASS_STARS}), cannot calibrate')
    setup_star(apass_df, coord)

    manual = set(MANUAL_REJECT.get(star_dir.name, []))
    mf = out_dir / 'manual_reject.txt'
    if mf.exists():
        manual |= {ln.strip() for ln in mf.read_text().splitlines() if ln.strip() and not ln.startswith('#')}

    total = len(plates)
    bar.max, bar.value = total, 0
    db = shelve.open(str(cache_dir / 'plate_db_shelf'), flag='c', writeback=False)
    start = time.monotonic()
    last_render, n_err, n_done_new = 0.0, 0, 0
    last_err = ''
    for i, name in enumerate(plates):
        cached = db.get(name)
        if not (cached and cached.get('_sig') == _SIG):
            label.value = f'{star_dir.name}: {name}'
            try:
                r = process_plate(star_dir / 'cutouts' / name)
            except Exception as e:
                r = {k: {'file': name, 'exp_id': exp_id_of(name), 'status': f'error: {e}'} for k in KINDS}
                n_err += 1
                last_err = str(e)
            r['_sig'] = _SIG
            db[name] = r
            n_done_new += 1
            if n_done_new == 50 and n_err > 5:
                db.sync(); db.close()
                raise RuntimeError(f'more than 5 of the first 50 plates failed with an error. Last error: {last_err}')
        now = time.monotonic()
        if now - last_render > 0.3 or (i + 1) == total:
            el = now - start
            rate = n_done_new / el if el > 0 and n_done_new else 0
            rem = (total - i - 1) / rate if rate > 0 else None
            bar.value = i + 1
            html.value = (f"<div style='font-family: monospace; font-size: 12px;'><b>{(i + 1) / total * 100:5.1f}%</b> &nbsp; {i + 1:,}/{total:,} plates"
                          f" &nbsp;|&nbsp; Elapsed {format_eta(el)} &nbsp;|&nbsp; ETA {format_eta(rem)}</div>")
            last_render = now
        if (i + 1) % 50 == 0:
            db.sync()
    db.sync()
    rows = {k: [db[n][k] for n in plates if n in db] for k in KINDS}
    db.close()

    A, B = tier_frame(rows['iso'], manual, True), tier_frame(rows['circ'], manual)
    m = merge_kinds(A, B).sort_values('date_jd').reset_index(drop=True)
    m.to_csv(out_dir / 'photometry_results.csv', index=False)

    pub = m[m['tier'].isin(PUB) & m['mag'].notna() & m['date_jd'].notna() & ~m['manual_reject']]
    lc = pd.DataFrame({'jd': pub['date_jd'], 'magnitude': pub['mag'], 'magnitude_error': pub['mag_err'], 'tier': pub['tier'],
                       'source': pub['source'], 'file': pub['file'], 'exp_id': pub['exp_id'], 'flags': pub['flags']}).reset_index(drop=True)
    lc.to_csv(out_dir / 'lightcurve.csv', index=False)
    lim = m[(m['tier'] == 'limit') & m['mag'].notna() & m['date_jd'].notna() & ~m['manual_reject']]
    lim[['date_jd', 'mag', 'tier', 'source', 'file', 'exp_id']].rename(columns={'date_jd': 'jd', 'mag': 'limit_mag'}).to_csv(out_dir / 'limits.csv', index=False)

    tiers = m['tier'].value_counts().to_dict()
    summ = {'star': star_dir.name, 'plates': len(m), 'points': len(lc), 'limits': len(lim), 'rejects': int(tiers.get('reject', 0)),
            'errors': int(m['status'].astype(str).str.startswith('error').sum()), 'from_circle': int((pub['source'] == 'circ').sum()),
            'from_iso': int((pub['source'] == 'iso').sum()), 'target_pos': coord_source, 'centre_offset_arcsec': round(float(off), 1) if off == off else np.nan, 'apass_src': apass_src}
    with open(out_dir / 'run_summary.txt', 'w') as fh:
        fh.write(f'SISaP run {datetime.now().isoformat(timespec="seconds")}  pipeline {PIPELINE_VERSION}\nPlates from: {plate_src}\n')
        fh.write(f'Target {coord.ra.deg:.6f} {coord.dec.deg:.6f} ({coord_source}); {off:.1f} arcsec from the centre of the cutouts\n')
        fh.write(f'Reference stars in field: {len(apass_df)} ({apass_src})\nTiers: {tiers}\nStatus: {m["status"].value_counts().to_dict()}\n')
        fh.write(f'WCS repair: repaired {int(m["f_wcs_applied"].sum())}, no offset found {int(m["f_wcs_unresolved"].sum())}\n')
    (out_dir / 'target_coord.txt').write_text(f'{coord.ra.deg:.9f} {coord.dec.deg:.9f} {coord_source}\n')
    summ['output_dir'] = str(out_dir)
    summ['plate_source'] = plate_src
    return summ

# RUN ALL STARS
star_dirs = sorted(d for d in BASE_DATA_DIR.iterdir() if d.is_dir() and (d / 'cutouts').is_dir() and not d.name.startswith('_'))
if STARS:
    star_dirs = [d for d in star_dirs if d.name in STARS]
print(f'{len(star_dirs)} star folders: ' + ', '.join(d.name for d in star_dirs))
if not star_dirs:
    print(f'No star folders with a cutouts/ directory in {BASE_DATA_DIR}. Run dasch_Cutout_Recreation first (or copy a star folder in).')

overall_bar = widgets.IntProgress(value=0, min=0, max=max(len(star_dirs), 1), description='Stars:')
overall_label = widgets.Label(value='Not started')
plate_bar = widgets.IntProgress(value=0, min=0, max=1, description='Plates:')
plate_html = widgets.HTML(value='')
plate_label = widgets.Label(value='')
display(widgets.VBox([widgets.HTML('<b>Overall progress across stars</b>'), widgets.HBox([overall_bar, overall_label]),
                      widgets.HTML("<b>Current star's plate progress</b>"), plate_bar, plate_html, plate_label]))

summaries, skipped = [], []
for i, star_dir in enumerate(star_dirs):
    overall_label.value = f'Resolving target for {star_dir.name}...'
    coord, src = resolve_target_coord(star_dir)
    if coord is None:
        print(f'SKIPPED {star_dir.name}: could not get a target position ({src}). Add it to MANUAL_TARGET_COORDS and run again.')
        skipped.append(star_dir.name)
        overall_bar.value = i + 1
        continue
    overall_label.value = f'Processing {star_dir.name} ({i + 1}/{len(star_dirs)}), position from: {src}'
    try:
        s = run_star(star_dir, coord, src, plate_bar, plate_html, plate_label)
        summaries.append(s)
        print(f"DONE {star_dir.name}: {s['plates']} plates -> {s['points']} light curve points, {s['limits']} limits, {s['rejects']} rejects, "
              f"{s['errors']} errors. Plates from {s['plate_source']}. APASS: {s['apass_src']}. Target position {s['target_pos']} ({s['centre_offset_arcsec']} arcsec from the cutout centre)")
    except Exception as e:
        print(f'SKIPPED {star_dir.name}: {e}')
        skipped.append(star_dir.name)
    overall_bar.value = i + 1
overall_label.value = f'Done. {len(summaries)} of {len(star_dirs)} stars processed.'

print('\n' + '=' * 70 + '\nSISaP SUMMARY\n' + '=' * 70)
if summaries:
    print(pd.DataFrame(summaries).drop(columns=['output_dir', 'plate_source', 'apass_src']).to_string(index=False))
if skipped:
    print(f'\nSkipped: {skipped}')

# Plots
def exp_id_col(d):
    return [f"{str(s).strip()}{int(p):05d}m{int(m)}s{int(sl)}" for s, p, m, sl in zip(d['series'], d['platenum'], d['mosnum'], d['solnum'])]

def ensure_jd(d):
    # newer daschlab files have no time_jd column, only 'time'
    if 'time_jd' in d.columns:
        return d
    if 'time' in d.columns:
        v = pd.to_numeric(d['time'], errors='coerce')
        if v.notna().mean() > 0.5:
            d['time_jd'] = v + 2400000.5 if v.median() < 1e5 else v
        else:
            t = pd.to_datetime(d['time'], errors='coerce')
            ok = t.notna()
            d['time_jd'] = np.nan
            if ok.any():
                d.loc[ok, 'time_jd'] = Time(t[ok].to_numpy(dtype='datetime64[ns]')).jd
    return d

def get_dasch(star_dir):
    out_dir = star_dir / '02_A'
    # the DASCH folder (made by dasch_Lightcurve_Extraction) first, then older files
    for f in (star_dir / 'DASCH' / f'{star_dir.name}_dasch_lightcurve_accepted.csv', out_dir / 'dasch_lightcurve.csv', star_dir / 'dasch_official_lightcurve.csv'):
        if f.exists():
            return ensure_jd(pd.read_csv(f)), f.name
    from daschlab import open_session
    sess = open_session(str(star_dir))
    tc = out_dir / 'target_coord.txt'
    if tc.exists():
        ra_, dec_ = [float(v) for v in tc.read_text().split()[:2]]
        sess.select_target(ra_deg=ra_, dec_deg=dec_)
    else:
        sess.select_target(star_dir.name.replace('_', ' '))
    sess.select_refcat('apass')
    lc = sess.lightcurve(0)
    lc.apply_standard_rejections()
    d = lc.to_pandas()
    try:
        if 'time_jd' not in d.columns:
            d['time_jd'] = np.asarray(lc['time'].jd, dtype=float)
    except Exception:
        pass
    d = ensure_jd(d)
    d.to_csv(out_dir / 'dasch_lightcurve.csv', index=False)
    return d, 'daschlab'

if MAKE_PLOTS:
    plot_dirs = sorted(d for d in BASE_DATA_DIR.iterdir() if d.is_dir() and (d / OUT_SUBDIR / 'lightcurve.csv').exists() and not d.name.startswith('_'))
    if STARS:
        plot_dirs = [d for d in plot_dirs if d.name in STARS]
    print(f'\n{len(plot_dirs)} stars with a SISaP light curve to plot')

    rows = []
    for star_dir in plot_dirs:
        out_dir = star_dir / OUT_SUBDIR
        sis = pd.read_csv(out_dir / 'lightcurve.csv')
        lim = pd.read_csv(out_dir / 'limits.csv') if (out_dir / 'limits.csv').exists() else pd.DataFrame()
        d, d_src = None, None
        try:
            d, d_src = get_dasch(star_dir)
            d = d[(d['reject'] == 0) & d['magcal_magdep'].notna() & d['time_jd'].notna()].copy()
        except Exception as e:
            print(f'  {star_dir.name}: no DASCH light curve ({e})')
            d = None
        err_col = None
        if d is not None:
            err_col = DASCH_ERR_COL if DASCH_ERR_COL in d.columns else ('magcal_local_rms' if 'magcal_local_rms' in d.columns else None)

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 11), sharex=True, sharey=True)
        ax1.errorbar(sis['jd'], sis['magnitude'], yerr=sis['magnitude_error'], fmt='none', ecolor='red', elinewidth=0.7, alpha=0.5, zorder=1)
        ax1.scatter(sis['jd'], sis['magnitude'], s=14, c='steelblue', alpha=0.8, zorder=2)
        if SHOW_LIMITS and len(lim):
            ax1.scatter(lim['jd'], lim['limit_mag'], s=26, marker='^', facecolors='none', edgecolors='steelblue', linewidths=0.9, zorder=2,
                        label=f'limit, brighter than this ({len(lim)})')
            ax1.legend(fontsize=8, loc='lower left')
        ax1.set_title(f'SISaP ({REF_CATALOG}): {star_dir.name.replace("_", " ")} ({len(sis)} points)', fontsize=9)
        if d is not None and len(d):
            if err_col is not None:
                ax2.errorbar(d['time_jd'], d['magcal_magdep'], yerr=d[err_col], fmt='none', ecolor='red', elinewidth=0.7, alpha=0.5, zorder=1)
            ax2.scatter(d['time_jd'], d['magcal_magdep'], s=14, c='black', alpha=0.8, zorder=2)
            ax2.set_title(f'DASCH: {star_dir.name.replace("_", " ")} ({len(d)} points' + (f', error bars = {err_col})' if err_col else ')'), fontsize=9)
        else:
            ax2.text(0.5, 0.5, 'DASCH light curve unavailable', ha='center', va='center', transform=ax2.transAxes)
        for ax in (ax1, ax2):
            ax.set_ylabel('Calibrated magnitude', fontsize=8)
            ax.grid(alpha=0.3)
        ax2.set_xlabel('Julian Date', fontsize=8)
        ax1.invert_yaxis()
        plt.tight_layout()
        png = out_dir / 'sisap_dasch_stacked.png'
        plt.savefig(png, dpi=130)
        plt.show()

        agree = ''
        if d is not None and len(d) and 'exp_id' in sis.columns:
            try:
                dm = pd.Series(d['magcal_magdep'].values, index=exp_id_col(d)).groupby(level=0).first()
                c = sis.assign(dasch=sis['exp_id'].map(dm)).dropna(subset=['dasch'])
                dl = c['magnitude'] - c['dasch']
                agree = f'{len(c)} plates in both, {(dl.abs() < 0.5).mean() * 100:.0f}% within 0.5 mag, median offset {dl.median():+.2f}'
            except Exception:
                agree = ''
        rows.append({'star': star_dir.name, 'sisap_points': len(sis), 'dasch_points': 0 if d is None else len(d), 'agreement (ours - DASCH)': agree})

    print('\nPLOT SUMMARY')
    if rows:
        print(pd.DataFrame(rows).to_string(index=False))

3 star folders: R_CrB, RY_Sgr, UW_Cen


DONE R_CrB: 387 plates -> 32 light curve points, 25 limits, 64 rejects, 0 errors. Plates from manifest dasch_accepted (387 listed, 0 not on disk). APASS: Gaia DR3 (GAVO): 1395 stars, 355 to G<18, 283 usable as calibrators. Target position cutout notebook (0.0 arcsec from the cutout centre)
DONE RY_Sgr: 689 plates -> 503 light curve points, 6 limits, 68 rejects, 0 errors. Plates from manifest dasch_accepted (689 listed, 0 not on disk). APASS: Gaia DR3 (GAVO): 17707 stars, 3263 to G<18, 1270 usable as calibrators. Target position cutout notebook (0.0 arcsec from the cutout centre)
